# Custom GluonTS Datasets

This notebook adapts the datasets used to train [TEMPO](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://arxiv.org/pdf/2310.04948) to a [GluonTS](https://ts.gluon.ai/stable/index.html) appropriate form and follows the [Quick Start Tutorial](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets).
- First, we'll create a custom GluonTS dataset that contains a single dataset
- Then, we'll create a custom GluonTS dataset that contains multiple datasets

## [Built-in Datasets](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Provided-datasets)

GluonTS comes with a number of publicly available built-in datasets. Here are the names of the datasets they provide.

In [46]:
from gluonts.dataset.repository import dataset_names

print(f"Available datasets: {dataset_names}")

Available datasets: ['constant', 'exchange_rate', 'solar-energy', 'electricity', 'traffic', 'exchange_rate_nips', 'electricity_nips', 'traffic_nips', 'solar_nips', 'wiki2000_nips', 'wiki-rolling_nips', 'taxi_30min', 'kaggle_web_traffic_with_missing', 'kaggle_web_traffic_without_missing', 'kaggle_web_traffic_weekly', 'm1_yearly', 'm1_quarterly', 'm1_monthly', 'nn5_daily_with_missing', 'nn5_daily_without_missing', 'nn5_weekly', 'tourism_monthly', 'tourism_quarterly', 'tourism_yearly', 'cif_2016', 'london_smart_meters_without_missing', 'wind_farms_without_missing', 'car_parts_without_missing', 'dominick', 'fred_md', 'pedestrian_counts', 'hospital', 'covid_deaths', 'kdd_cup_2018_without_missing', 'weather', 'm3_monthly', 'm3_quarterly', 'm3_yearly', 'm3_other', 'm4_hourly', 'm4_daily', 'm4_weekly', 'm4_monthly', 'm4_quarterly', 'm4_yearly', 'm5', 'uber_tlc_daily', 'uber_tlc_hourly', 'airpassengers', 'australian_electricity_demand', 'electricity_hourly', 'electricity_weekly', 'rideshare_wit

We can download one of the built-in datasets using the [`get_dataset()`](https://ts.gluon.ai/stable/api/gluonts/gluonts.dataset.repository.html?highlight=get_dataset#gluonts.dataset.repository.get_dataset) method.

In [47]:
from gluonts.dataset.repository import get_dataset

dataset = get_dataset("m4_hourly")
print(f"dataset type: {type(dataset)}")

dataset type: <class 'gluonts.dataset.common.TrainDatasets'>


GluonTS [`TrainDatasets`](https://ts.gluon.ai/stable/api/gluonts/gluonts.dataset.common.html?highlight=traindataset#gluonts.dataset.common.TrainDatasets) are objects that represent all the datasets for training and testing. A `TrainDatasets` instance has three main members:
- `dataset.train` 
- `dataset.test`
- `dataset.metadata`

`dataset.train` is an iterable collection of time series sequences used for training.
- Each entry is a dictionary that corresponds to a single time series and containing the following key-value pairs:
    - `target`: An array of time series values
    - `start`: The starting timestamp of the time series.

In [48]:
from gluonts.dataset.util import to_pandas

print(f"dataset.train type: {type(dataset.train)}")

# Create an iterator over dataset.train
iterator = iter(dataset.train)
print(f"Number of time series: {len(dataset.train)}")

# Get the first time series in dataset.train
first_time_series = next(iterator)
print(f"first_time_series type: {type(first_time_series)}")
print(f"first_time_series keys: {first_time_series.keys()}")

# Convert the first time series into a Pandas series
train_series = to_pandas(first_time_series)
print(f"train_series type: {type(train_series)}")
print(f"Length of each time series: {len(train_series)}")

# Get the first time series's values
target = first_time_series["target"]
print(f"target[:10] = {target[:10]}")

# Get the first time series's starting timestamp
start = first_time_series["start"]
print(f"start = {start}")

dataset.train type: <class 'gluonts.itertools.Map'>
Number of time series: 414
first_time_series type: <class 'dict'>
first_time_series keys: dict_keys(['target', 'start', 'feat_static_cat', 'item_id'])
train_series type: <class 'pandas.core.series.Series'>
Length of each time series: 700
target[:10] = [605. 586. 586. 559. 511. 443. 422. 395. 382. 370.]
start = 1750-01-01 00:00


Similar to `dataset.train`, `dataset.test` is an iterable collection of data entries used for inference.
- Each entry in `dataset.test` is an extended version of the corresponding entry in `dataset.train`, containing additional time steps at the end of the series. 
- This extension, known as the forecasting window, has a length equal to the recommended prediction length and represents the period that the model aims to forecast.

In [49]:
# Create an iterator over dataset.test
test_iterator = iter(dataset.test)

# Get the first time series in dataset.test
first_time_series = next(test_iterator)

# Convert the first time series into a Pandas series
test_series = to_pandas(first_time_series)

# Get the first time series's values
values = first_time_series["target"]
print(f"target[:10] = {values[:10]}")

# Get the first time series's starting timestamp
start = first_time_series["start"]
print(f"start = {start}")

# Get the forecasting window's length for the first time series
forecasting_window_length = len(test_series) - len(train_series)
print(f"Forecasting window length: {forecasting_window_length}")

target[:10] = [605. 586. 586. 559. 511. 443. 422. 395. 382. 370.]
start = 1750-01-01 00:00
Forecasting window length: 48


- `dataset.metadata` contains metadata of the dataset such as the frequency of the time series, a recommended prediction horizon, associated features, etc.

In [50]:
# Number of future time steps to predict values for
prediction_length = dataset.metadata.prediction_length
print(f"Prediction length: {prediction_length}")

# How often values are recorded in the time series
frequency = dataset.metadata.freq
print(f"Frequency: {frequency}")

Prediction length: 48
Frequency: H


## [Custom Datasets](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets)

GluonTS doesn't require custom datasets to use the same format as built-in datasets. The only requirements for a custom dataset are
1. be iterable
2. have a `target` and `start` field

### [Dummy Dataset](https://ts.gluon.ai/stable/tutorials/forecasting/quick_start_tutorial.html#Custom-datasets)

Here's an example of creating a custom GluonTS dataset using randomly generated data. The dataset's values (i.e. the values for each time series) will be in a `numpy.array` and the dataset's indices (i.e. timestamps) will be in a [`pandas.Period`](https://pandas.pydata.org/docs/reference/api/pandas.Period.html).

First, we'll define the dataset's metadata and create its values and indices.
- Each row is a different time series
- Each column is a different time stamp

In [51]:
import numpy as np
import pandas as pd

# Number of time series to store in the dummy dataset
num_time_series = 256
print(f"Number of time series: {num_time_series}")

# Number of time steps in each time series
num_time_steps = 336
print(f"Number of steps in each time series: {num_time_steps}")

# Define each time series's frequency
freq = "1H"

# Create randomy generated values to use in the dummy dataset
dummmy_values = np.random.normal(size=(num_time_series, num_time_steps))

# Create the starting timestamp of the dummy dataset
start = pd.Period("01-01-2019", freq=freq)

print(f"dummy_values.shape: {dummmy_values.shape}")
print(f"start: {start}")  # can be different for each dataset

Number of time series: 256
Number of steps in each time series: 336
dummy_values.shape: (256, 336)
start: 2019-01-01 00:00


Then, we'll split the dataset into the training and test splits, and create a [`ListDataset`](https://ts.gluon.ai/stable/api/gluonts/gluonts.dataset.common.html?highlight=listdataset#gluonts.dataset.common.ListDataset) for training and a ListDataset for testing. And that's it! Our dataset is now in a GluonTS appropriate form. 

In [52]:
from gluonts.dataset.common import ListDataset

# Remove the prediction window from the data in the training set
train_data = dummmy_values[:, :-prediction_length]

# Create a list where each element is a key-value pair mapping starting
# timestamp to time series values
train_datasets = [{"start": start, "target": time_series} for time_series in train_data]

# Create the training set
train_set = ListDataset(train_datasets, freq=freq)

# Create the test set and include the prediction window
test_datasets = [
    {"target": time_series, "start": start} for time_series in dummmy_values
]

test_set = ListDataset(test_datasets, freq=freq)

### Custom Datasets for TEMPO

Now that we know how to create a custom GluonTS dataset using dummy data, let's create a custom GluonTS dataset using the data we want to train TEMPO on.

First, let's create a list contains all the time series sequences we want to train TEMPO on.  

In [ ]:
# Each element is a list of time series values
main_list = []

#### Adapting a Single Dataset to GluonTS

First, we'll create a custom GluonTS dataset that contains a single dataset.

#### Adapting Multiple Datasets to GluonTS